# RAG + LLM Evaluation Runner

This notebook mirrors `naive_llm.ipynb`, but augments each PR prompt with top-k repo-aware guideline/review chunks retrieved through the S15 adaptive rerank strategy in `src/rag_model/s15_query_strategy.py`.

In [1]:
import os, json, time
from dotenv import load_dotenv
from pathlib import Path
from groq import Groq
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path("..").resolve()
EVAL_PATH = ROOT / 'data' / 'processed' / 'evaluation.json'
OUT_DIR = ROOT / 'outputs'
RAW_OUT = OUT_DIR / 'rag_llm_raw_responses_v2.txt'
ZERO_RESPONSE_PR = OUT_DIR / 'zero_response_PRs_rag_v2.txt'
PARSED_OUT = OUT_DIR / 'llm_reviews.json'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = 'openai/gpt-oss-20b'
BATCH_SIZE = 1
MAX_CHARS_PER_FILE = 8000

# evaluation.json range controls (inclusive, 0-based)
STARTING = 0
ENDING = 96

# Token controls
MODEL_CONTEXT_LIMIT = 8192
MAX_OUTPUT_TOKENS = 800
INPUT_TOKEN_BUDGET = int(MODEL_CONTEXT_LIMIT * 0.75) - MAX_OUTPUT_TOKENS
CHARS_PER_TOKEN = 2.5

# Retrieval controls
RETRIEVAL_TOP_K = 5
PROMPT_PATH = ROOT / 'src' / 'rag_model' / 'prompts' / 'v1.txt'
PROMPT_TEMPLATE = PROMPT_PATH.read_text(encoding='utf-8')

load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY_V2']
GROQ_API_URL = os.environ.get('GROQ_API_URL', 'https://api.groq.com')
client = Groq(api_key=GROQ_API_KEY, base_url=GROQ_API_URL)

In [2]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from rag_model.s15_query_strategy import s15_adapt_rerank_refined

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_evaluation(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return [r for r in data if isinstance(r, dict) and r.get('source_file')]

def resolve_source_path(source_file: str):
    p = Path(source_file)
    if p.is_absolute() and p.exists():
        return p
    p1 = ROOT / source_file
    if p1.exists():
        return p1
    p2 = ROOT / 'data' / 'processed' / source_file
    if p2.exists():
        return p2
    p3 = ROOT / 'data' / 'processed' / 'evaluation_files' / p.name
    return p3

def read_source_text(source_file: str):
    p = resolve_source_path(source_file)
    text = p.read_text(encoding='utf-8', errors='ignore')
    if len(text) <= MAX_CHARS_PER_FILE:
        return text
    half = MAX_CHARS_PER_FILE // 2
    return text[:half] + '\n\n...TRUNCATED...\n\n' + text[-half:]

def est_tokens(text: str):
    return max(1, int(len(text) / CHARS_PER_TOKEN))

def build_s15_context(entry: dict):
    candidates = s15_adapt_rerank_refined(
        entry['source_code'],
        entry.get('repo'),
        top_k=RETRIEVAL_TOP_K,
    )
    chunks = []
    for candidate in candidates:
        chunk = str(candidate.get('text', '')).strip()
        if chunk:
            chunks.append(chunk)
    return 'S15 adaptive rerank retrieval', chunks, candidates

def save_s15_cache(prepared, cache_path):
    """Save prepared entries to cache file."""
    cache_data = []
    for entry in prepared:
        cache_data.append({
            'id': entry['id'],
            'query_text': entry['query_text'],
            'retrieved_chunks': entry['retrieved_chunks'],
            'retrieved_candidates': entry['retrieved_candidates']
        })
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache_data, f, indent=2)

def load_s15_cache(cache_path):
    """Load prepared entries from cache, return dict by id."""
    if not cache_path.exists():
        return {}
    with open(cache_path, 'r', encoding='utf-8') as f:
        cache_data = json.load(f)
    return {item['id']: item for item in cache_data}

records = load_evaluation(EVAL_PATH)
records = records[STARTING:ENDING + 1]
print('selected_records:', len(records), 'range:', STARTING, 'to', ENDING, '(inclusive)')

selected_records: 97 range: 0 to 96 (inclusive)


In [4]:
def build_prompt(batch):
    header = PROMPT_TEMPLATE.strip()

    blocks = [header]
    for item in batch:
        chunks = item.get('retrieved_chunks', [])
        chunk_block = '\n\n'.join([f'[Chunk {idx+1}]\n{c}' for idx, c in enumerate(chunks)])
        if not chunk_block:
            chunk_block = '[No retrieved payload chunks available]'

        blocks.append(
f"""RETRIEVED_PAYLOAD_CHUNKS:
{chunk_block}
END

PR:
ID: {item['id']}
CODE:
{item['source_code']}
"""
        )

    return '\n'.join(blocks)

def call_groq(prompt):
    return client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )

In [5]:
# Single-sample run (no file writes)
sample = records[0]
entry = {
    'id': sample['id'],
    'repo': sample.get('repo'),
    'source_file': sample['source_file'],
    'source_code': read_source_text(sample['source_file'])
}
entry['query_text'], entry['retrieved_chunks'], entry['retrieved_candidates'] = build_s15_context(entry)
prompt = build_prompt([entry])
print('=== QUERY TEXT ===')
print(entry['query_text'])
print('\\n=== RETRIEVED CHUNK COUNT ===', len(entry['retrieved_chunks']))
print('\\n=== PROMPT (truncated 2500 chars) ===')
print(prompt[:2500])
print('\\n=== CALLING GROQ ===')
resp = call_groq(prompt)
content = resp.choices[0].message.content
print('\\n=== RESPONSE ===')
print(content)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1764.31it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== QUERY TEXT ===
S15 adaptive rerank retrieval
\n=== RETRIEVED CHUNK COUNT === 5
\n=== PROMPT (truncated 2500 chars) ===
TASK: Detect STRICT PEP 8-style violations in Python code.

PROCESSING RULES:
- Process EACH PR independently.
- After finishing ONE PR, immediately output its JSON object.
- Do NOT wait for other PRs.
- Do NOT analyze across PRs.

VIOLATION TYPES (ONLY these):

1) naming_convention:
   - Functions, variables, parameters → must be snake_case
   - Classes → must be PascalCase (CapWords)
   - Constants → must be UPPER_CASE
   - Flag:
       * camelCase names in functions/variables/params
       * PascalCase used for functions/variables
       * snake_case used for class names

2) indentation:
   - Indentation must be exactly 4 spaces per level
   - Flag:
       * tabs used
       * non-multiple of 4 spaces
       * inconsistent indentation within block

3) unused_import:
   - Imported module/symbol not referenced anywhere in file

4) mutable_default:
   - Function pa

In [11]:
print(prompt)

TASK: Detect STRICT PEP 8-style violations in Python code.

PROCESSING RULES:
- Process EACH PR independently.
- After finishing ONE PR, immediately output its JSON object.
- Do NOT wait for other PRs.
- Do NOT analyze across PRs.

VIOLATION TYPES (ONLY these):

1) naming_convention:
   - Functions, variables, parameters → must be snake_case
   - Classes → must be PascalCase (CapWords)
   - Constants → must be UPPER_CASE
   - Flag:
       * camelCase names in functions/variables/params
       * PascalCase used for functions/variables
       * snake_case used for class names

2) indentation:
   - Indentation must be exactly 4 spaces per level
   - Flag:
       * tabs used
       * non-multiple of 4 spaces
       * inconsistent indentation within block

3) unused_import:
   - Imported module/symbol not referenced anywhere in file

4) mutable_default:
   - Function parameters using [] or {} as default values

5) documentation_formatting:
   - Docstring indentation inconsistent with block 

In [6]:
batch_size = BATCH_SIZE

# Setup caching
S15_CACHE_PATH = OUT_DIR / f's15_cache_top_k_{RETRIEVAL_TOP_K}.json'
cached_s15 = load_s15_cache(S15_CACHE_PATH)
print(f'Loaded {len(cached_s15)} cached S15 results')

# Parallel S15 retrieval with caching
prepared = []
records_to_fetch = []
for r in records:
    if r['id'] in cached_s15:
        # Use cached result
        cached = cached_s15[r['id']]
        entry = {
            'id': r['id'],
            'repo': r.get('repo'),
            'source_file': r['source_file'],
            'source_code': read_source_text(r['source_file']),
            'query_text': cached['query_text'],
            'retrieved_chunks': cached['retrieved_chunks'],
            'retrieved_candidates': cached['retrieved_candidates']
        }
        prepared.append(entry)
    else:
        records_to_fetch.append(r)

print(f'Using {len(prepared)} cached entries, fetching {len(records_to_fetch)} new entries')

# Parallel fetch for uncached records
if records_to_fetch:
    with ThreadPoolExecutor(max_workers=4) as executor:
        future_to_record = {}
        for r in records_to_fetch:
            entry_partial = {
                'id': r['id'],
                'repo': r.get('repo'),
                'source_file': r['source_file'],
                'source_code': read_source_text(r['source_file'])
            }
            future = executor.submit(build_s15_context, entry_partial)
            future_to_record[future] = (r, entry_partial)
        
        for i, future in enumerate(as_completed(future_to_record)):
            r, entry_partial = future_to_record[future]
            try:
                qtext, chunks, candidates = future.result()
                entry_partial['query_text'] = qtext
                entry_partial['retrieved_chunks'] = chunks
                entry_partial['retrieved_candidates'] = candidates
                prepared.append(entry_partial)
                if (i + 1) % 10 == 0:
                    print(f'  Completed {i + 1}/{len(records_to_fetch)} S15 retrievals')
            except Exception as e:
                entry_partial['query_text'] = f'<query/retrieval error: {e}>'
                entry_partial['retrieved_chunks'] = []
                entry_partial['retrieved_candidates'] = []
                prepared.append(entry_partial)
                print(f'  Error on record {r["id"]}: {e}')
    
    # Save results to cache
    save_s15_cache(prepared, S15_CACHE_PATH)
    print(f'Saved S15 results to cache')

id_to_repo = {x['id']: x['repo'] for x in prepared}
zero_response_prs = set()

i = 0
batch_no = 0
while i < len(prepared):
    time.sleep(2)
    batch = []
    while i < len(prepared) and len(batch) < batch_size:
        batch.append(prepared[i])
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)
    resp = call_groq(prompt)
    content = resp.choices[0].message.content

    print(f'\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    pr_ids = [p['id'] for p in batch]
    print(f'PR ids: {pr_ids}')
    print(f'content_type: {type(content).__name__}')

    is_empty = False
    if isinstance(content, str):
        content_stripped = content.strip()
        print(f'content_len: {len(content)}')
        print(f'is_empty_after_strip: {len(content_stripped) == 0}')
        print('response_preview:')
        print(content[:1200])
        if len(content_stripped) == 0:
            is_empty = True
    else:
        print('response_preview_non_str:')
        print(content)
        is_empty = True

    if is_empty:
        print(f'Empty response detected for batch {batch_no}')
        zero_response_prs.update(pr_ids)

    content_to_write = content if isinstance(content, str) else str(content)

    # Keep exact header/body format used in naive_llm
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {pr_ids}\n')
        rf.write(f'repos: {[id_to_repo[p_id] for p_id in pr_ids]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f'Batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}')

if zero_response_prs:
    with open(ZERO_RESPONSE_PR, 'w', encoding='utf-8') as f:
        for pr_id in sorted(zero_response_prs):
            f.write(f'{pr_id}\n')
    print(f'Wrote {len(zero_response_prs)} zero-response PR ids to {ZERO_RESPONSE_PR}')
else:
    print('No zero-response PRs detected.')

Loaded 0 cached S15 results
Using 0 cached entries, fetching 97 new entries
  Completed 10/97 S15 retrievals
  Completed 20/97 S15 retrievals
  Completed 30/97 S15 retrievals
  Completed 40/97 S15 retrievals
  Completed 50/97 S15 retrievals
  Completed 60/97 S15 retrievals
  Completed 70/97 S15 retrievals
  Completed 80/97 S15 retrievals
  Completed 90/97 S15 retrievals
Saved S15 results to cache

=== BATCH 1 RESPONSE DIAGNOSTICS ===
PR ids: ['synthetic-django_PR_24']
content_type: str
content_len: 606
is_empty_after_strip: False
response_preview:
[
  {
    "PR_ID": "synthetic-django_PR_24",
    "llm_reviews": [
      {
        "line_number": 31,
        "violation_category": "naming_convention",
        "review_comment": "Method name 'byAuthor' violates PEP8; use snake_case 'by_author'."
      },
      {
        "line_number": 31,
        "violation_category": "mutable_default",
        "review_comment": "Parameter 'author_id' uses mutable default []; use None or a default int."
     